In [8]:
# package import + setup
import re
from pathlib import Path

import pandas as pd

TARGET_COLS = [
    'source_file',
    'denomination',
    'record_id',
    'first_name',
    'last_name',
    'job_title',
    'email',
    'mobile_phone',
    'company_linkedin',
    'facebook',
    'twitter',
    'work_phone',
    'industry',
    'company_name',
    'company_website',
    'company_address',
    'company_zipcode',
    'company_employee_size_actual',
    'company_city',
    'company_state',
    'company_revenue',
    'company_location',
    'company_founded_at',
    'gender',
    'company_zipfour',
    'county',
    'company_description',
    'primary_sic_code',
    'primary_sic_code_description',
    'primary_naics',
    'primary_naics_description',
    'cuisine_code',
    'cuisine_code_description',
    'location_sales_volume_range',
    'location_sales_volume_actual',
    'company_employee_size_range',
    'company_sales_volume_range',
    'company_sales_volume_actual',
    'business_type',
    'credit_cards_accepted',
    'linkedin',
    'landline_phone',
    'home_address',
    'home_city',
    'home_state',
    'home_zipcode',
    'state_voter_id',
    'age',
    'age_range',
    'party_description',
    'ethnic_group',
    'us_congressional_district',
    'state_senate_district',
    'state_legislative_district',
    'state_house_district',
    'precinct',
    'county_commissioner_district',
    'county_supervisorial_district',
    'language_code',
    'marital_status',
    'religion_code',
    'presence_of_children_in_household',
    'household_net_worth',
    'veteran_in_household',
    'voting_performance_even_year_general',
    'voting_performance_even_year_primary',
    'voting_performance_even_year_general_and_primary',
    'voting_performance_minor_election',
    'primary_n_of_4',
    'general_2024',
    'primary_2024',
    'general_2022',
    'primary_2022',
    'general_2020',
    'primary_2020',
]

CHURCH_RAW_DIR = Path("raw_data_messy/church")


In [9]:
# --- reusable helpers (church ingest) ---

ZIP_RE_END = re.compile(r"(\d{5})(?:-(\d{4}))?\s*$")
STATE_ABBR = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "DC"
}
STATE_NAME_TO_ABBR = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR", "CALIFORNIA": "CA", "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE", "FLORIDA": "FL", "GEORGIA": "GA", "HAWAII": "HI", "IDAHO": "ID", "ILLINOIS": "IL", "INDIANA": "IN", "IOWA": "IA", "KANSAS": "KS", "KENTUCKY": "KY", "LOUISIANA": "LA", "MAINE": "ME", "MARYLAND": "MD", "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN", "MISSISSIPPI": "MS", "MISSOURI": "MO", "MONTANA": "MT", "NEBRASKA": "NE", "NEVADA": "NV", "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ", "NEW MEXICO": "NM", "NEW YORK": "NY", "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH", "OKLAHOMA": "OK", "OREGON": "OR", "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC", "SOUTH DAKOTA": "SD", "TENNESSEE": "TN", "TEXAS": "TX", "UTAH": "UT", "VERMONT": "VT", "VIRGINIA": "VA", "WASHINGTON": "WA", "WEST VIRGINIA": "WV", "WISCONSIN": "WI", "WYOMING": "WY", "DISTRICT OF COLUMBIA": "DC"
}
STATE_NAME_RE = re.compile(r"\b(" + "|".join(sorted(STATE_NAME_TO_ABBR.keys(), key=len, reverse=True)) + r")\b", flags=re.IGNORECASE)
STATE_ABBR_RE = re.compile(r"\b(" + "|".join(sorted(STATE_ABBR)) + r")\b")
CITY_TOKEN_RE = re.compile(r"^[A-Za-z][A-Za-z\.'-]*$")
STREET_SUFFIX_TOKENS = {
    "ALY", "AVE", "BLVD", "CIR", "CT", "CV", "DR", "EXPY", "FWY", "HWY", "LN", "LP", "PL", "PKWY", "RD", "RTE", "SQ", "ST", "TER", "TRL", "WAY",
    "ALLEY", "AVENUE", "BOULEVARD", "CIRCLE", "COURT", "COVE", "DRIVE", "EXPRESSWAY", "FREEWAY", "HIGHWAY", "LANE", "LOOP", "PLACE", "PARKWAY", "ROAD", "ROUTE", "SQUARE", "STREET", "TERRACE", "TRAIL"
}
DIRECTION_TOKENS = {"N", "S", "E", "W", "NE", "NW", "SE", "SW"}


def is_missing(x) -> bool:
    return x is None or pd.isna(x)


# After a leading √ (almost never intentional in this dataset), strip the usual cp1252 junk run.
_MOJIBAKE_AFTER_SQRT = frozenset(
    "\u221a\u00a2\u201a\u00c7\u00a8\u00c4\u03c0\u00b0, \t"
)  # √ ¢ ‚ Ç ¨ Ä π ° plus comma/space
_MOJIBAKE_HINT_RE = re.compile(r"[âÃƒÂ]|â€|Ã¢")


def maybe_repair_mojibake(s: str) -> str:
    """Fix common Excel/cp1252 vs UTF-8 mismatches (garbled prefix or â€˜ style text)."""
    if not s:
        return s
    if s.startswith("\u221a"):
        i = 0
        while i < len(s) and s[i] in _MOJIBAKE_AFTER_SQRT:
            i += 1
        s = s[i:]
    if _MOJIBAKE_HINT_RE.search(s):
        try:
            t = s.encode("cp1252").decode("utf-8")
            if t != s:
                return t
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass
    return s


def normalize_text_value(value):
    if is_missing(value):
        return pd.NA
    s = str(value).replace("\u202f", " ").replace("\u00a0", " ")
    # SpreadsheetML / Excel exports sometimes embed literal _x000D_ (CR) etc. in cell text.
    s = re.sub(r"_x[0-9a-f]{4}_", "", s, flags=re.IGNORECASE)
    s = re.sub(r"_x[0-9a-f]{4}$", "", s, flags=re.IGNORECASE)
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"[\x00-\x1f\x7f]", "", s)
    s = re.sub(r"[\ufeff\u200b-\u200f\u202a-\u202e\u2060-\u2064]+", "", s)
    s = maybe_repair_mojibake(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else pd.NA


def format_us_phone_value(value):
    """Normalize US phone to (XXX) XXX-XXXX when 10 digits are present."""
    s = normalize_text_value(value)
    if is_missing(s):
        return pd.NA
    s = str(s)
    digits = re.sub(r"\D", "", s)
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    if len(digits) == 10:
        return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"
    return s


def normalize_email_value(value):
    """Strip all whitespace from email (fixes pasted breaks and 'user _name@' typos)."""
    s = normalize_text_value(value)
    if is_missing(s):
        return pd.NA
    s = re.sub(r"\s+", "", str(s))
    return s if s else pd.NA


def clean_contact_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "email" in df.columns:
        df["email"] = df["email"].map(normalize_email_value)
    for col in ("work_phone", "mobile_phone", "landline_phone"):
        if col in df.columns:
            df[col] = df[col].map(format_us_phone_value)
    return df


def normalize_string_columns(df: pd.DataFrame, columns=None) -> pd.DataFrame:
    df = df.copy()
    if columns is None:
        columns = [
            c for c in df.columns
            if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
        ]
    for c in columns:
        if c in df.columns:
            df[c] = df[c].map(normalize_text_value)
    return df


def split_person_name(name):
    """Split full name into (first_name, last_name): last token is last name, everything before is first_name."""
    s = normalize_text_value(name)
    if is_missing(s):
        return None, None
    s = re.sub(r"^[.\s,;:]+", "", str(s)).strip()
    s = re.sub(r"^(?:(?:dr|doctor)\.?\s+)+", "", str(s), flags=re.IGNORECASE).strip()
    if not s:
        return None, None
    parts = s.split(" ")
    if len(parts) == 1:
        return parts[0], None
    return " ".join(parts[:-1]), parts[-1]


def rename_keep_only(df: pd.DataFrame, rename_map: dict) -> pd.DataFrame:
    """Strip headers, rename, keep only mapped columns (drops everything else)."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    ex = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=ex)
    keep = list(dict.fromkeys(ex.values()))
    return df[[c for c in keep if c in df.columns]]


def drop_fully_empty_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Drop rows where every column is blank/NA at ingestion time."""
    df = normalize_string_columns(df)
    return df.dropna(how="all")


def add_first_last_from_full_column(df: pd.DataFrame, full_col: str) -> pd.DataFrame:
    """Split one full-name column into first_name / last_name; drop full_col."""
    df = df.copy()
    if full_col not in df.columns:
        df["first_name"] = pd.NA
        df["last_name"] = pd.NA
        return df
    t = df[full_col].map(split_person_name)
    df["first_name"] = t.map(lambda z: z[0] if z[0] else pd.NA)
    df["last_name"] = t.map(lambda z: z[1] if z[1] else pd.NA)
    return df.drop(columns=[full_col])


def _extract_city_from_prefix(prefix):
    prefix = normalize_text_value(prefix)
    if is_missing(prefix):
        return None

    prefix = str(prefix).strip(" ,")
    if not prefix:
        return None

    if "," in prefix:
        city = prefix.split(",")[-1].strip(" ,")
        return city or None

    tokens = [t.strip(" ,") for t in prefix.split(" ") if t.strip(" ,")]
    if not tokens:
        return None

    city_tokens = []
    i = len(tokens) - 1
    while i >= 0:
        token = tokens[i].strip(" ,")
        token_upper = token.upper().rstrip(".")

        if not CITY_TOKEN_RE.match(token):
            break
        if token_upper in STREET_SUFFIX_TOKENS:
            break
        if token_upper in DIRECTION_TOKENS and city_tokens:
            break

        city_tokens.append(token)
        if len(city_tokens) >= 4:
            break
        i -= 1

    if not city_tokens:
        return None

    return " ".join(reversed(city_tokens)).strip() or None


def parse_address(addr):
    if is_missing(addr):
        return None, None, None, None

    s = normalize_text_value(addr)
    if is_missing(s):
        return None, None, None, None

    s = str(s)
    s = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", s)
    s = re.sub(r"(?<=\d)(?=[A-Za-z])", " ", s)
    s = re.sub(r"\b(?:UNITED\s+STATES|USA)\b\.?", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*,\s*", ", ", s).strip(" ,")

    z5 = z4 = None
    m_zip = ZIP_RE_END.search(s)
    if m_zip:
        z5 = m_zip.group(1)
        z4 = m_zip.group(2)
        s = s[: m_zip.start()].strip(" ,")

    state = None
    state_start = None

    m_abbr = None
    for m in STATE_ABBR_RE.finditer(s.upper()):
        m_abbr = m
    if m_abbr:
        state = m_abbr.group(1)
        state_start = m_abbr.start()
    else:
        m_name = None
        for m in STATE_NAME_RE.finditer(s):
            m_name = m
        if m_name:
            state = STATE_NAME_TO_ABBR[m_name.group(1).upper()]
            state_start = m_name.start()

    city = None
    if state_start is not None:
        prefix = s[:state_start].rstrip(" ,")
        city = _extract_city_from_prefix(prefix)

    return city or None, state, z5, z4


def add_parsed_address_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "company_address" not in df.columns:
        return df
    tup = df["company_address"].map(parse_address)
    df["company_city"] = tup.map(lambda t: t[0] if t and t[0] else pd.NA)
    df["company_state"] = tup.map(lambda t: t[1] if t and t[1] else pd.NA)
    df["company_zipcode"] = tup.map(lambda t: t[2] if t and t[2] else pd.NA)
    df["company_zipfour"] = tup.map(lambda t: t[3] if t and t[3] else pd.NA)
    return df


def _strip_trailing_component(address: str, component) -> str:
    if not address or is_missing(component):
        return address
    comp = str(component).strip(" ,")
    if not comp:
        return address
    pattern = rf"(?:,\s*|\s+){re.escape(comp)}\s*$"
    return re.sub(pattern, "", address, flags=re.IGNORECASE)


def clean_company_address_components(df: pd.DataFrame) -> pd.DataFrame:
    """Trim duplicated city/state/zip from company_address while preserving street detail."""
    df = df.copy()
    if "company_address" not in df.columns:
        return df

    state_abbr_to_name = {v: k.title() for k, v in STATE_NAME_TO_ABBR.items()}

    def clean_one(row):
        original = normalize_text_value(row.get("company_address"))
        if is_missing(original):
            return pd.NA

        cleaned = str(original)
        components = [
            row.get("company_zipcode"),
            row.get("company_zipfour"),
            row.get("company_state"),
            state_abbr_to_name.get(str(row.get("company_state")).upper(), pd.NA),
            row.get("company_city"),
        ]

        country_tail_re = r"(?:^|[\s,])(?:UNITED\s+STATES|USA|US|ESTADOS\s+UNIDOS|EE\.?\s*UU\.?)\.?\s*$"
        zip_tail_re = r"(?:\d{5}(?:-\d{4})?)"

        city_value = normalize_text_value(row.get("company_city"))
        city_re = re.escape(str(city_value)) if not is_missing(city_value) else None

        state_tokens = []
        state_abbr = normalize_text_value(row.get("company_state"))
        state_name = normalize_text_value(state_abbr_to_name.get(str(row.get("company_state")).upper(), pd.NA))
        if not is_missing(state_abbr):
            state_tokens.append(re.escape(str(state_abbr)))
        if not is_missing(state_name):
            state_tokens.append(re.escape(str(state_name)))
        state_re = "(?:" + "|".join(dict.fromkeys(state_tokens)) + ")" if state_tokens else None

        # Iteratively strip trailing geography tokens so mixed-order tails like
        # ", United States, Alabama" are fully removed without touching street text.
        previous = None
        passes = 0
        while cleaned != previous and passes < 8:
            previous = cleaned

            cleaned = re.sub(country_tail_re, "", cleaned, flags=re.IGNORECASE).strip(" ,=")
            for comp in components:
                cleaned = _strip_trailing_component(cleaned, comp).strip(" ,=")

            if city_re and state_re:
                cleaned = re.sub(
                    rf"(?:,|\s)+{city_re}\s*,?\s*{state_re}\.?(?:\s*,?\s*{zip_tail_re})?\s*(?:,?\s*(?:UNITED\s+STATES|USA|US|ESTADOS\s+UNIDOS|EE\.?\s*UU\.?)\.?)?\s*$",
                    "",
                    cleaned,
                    flags=re.IGNORECASE,
                ).strip(" ,=")

            cleaned = re.sub(rf",?\s*{zip_tail_re}\s*$", "", cleaned).strip(" ,=")
            cleaned = re.sub(country_tail_re, "", cleaned, flags=re.IGNORECASE).strip(" ,=")
            passes += 1

        cleaned = re.sub(r"\s*,\s*", ", ", cleaned).strip(" ,=")
        cleaned = re.sub(r"\s+", " ", cleaned).strip()

        if not cleaned:
            return original

        cleaned_alnum = re.sub(r"[^A-Za-z0-9]", "", cleaned)
        original_alnum = re.sub(r"[^A-Za-z0-9]", "", str(original))
        if len(cleaned_alnum) < 5 and len(original_alnum) >= 5:
            return original

        return cleaned

    df["company_address"] = df.apply(clean_one, axis=1)
    return df


In [10]:
# --- one-off: Denominations faculty C list delivery (hard-coded path) ---

PATH_CHURCH_LIST = CHURCH_RAW_DIR / "Denominations  faculty C list in usa  - Delivery of 4_3_2026 (17,313).xlsx"
if not PATH_CHURCH_LIST.is_file():
    raise FileNotFoundError(PATH_CHURCH_LIST)

RENAME_CHURCH_LIST = {
    "church-list-item__title": "company_name",
    "Location": "company_location",
    "Church address": "company_address",
    "Website": "company_website",
    "Phone": "work_phone",
    "Title": "job_title",
    "Company Email": "email",
    "Personal phone": "mobile_phone",
    "Persons name": "__full_name",
}

# Only one sheet in this workbook, so read it directly
part_church_list = pd.read_excel(PATH_CHURCH_LIST, dtype="string")
part_church_list = rename_keep_only(part_church_list, RENAME_CHURCH_LIST)
part_church_list = drop_fully_empty_rows(part_church_list)
part_church_list = add_first_last_from_full_column(part_church_list, "__full_name")
part_church_list["source_file"] = PATH_CHURCH_LIST.name
part_church_list["denomination"] = True

print("part_church_list", part_church_list.shape)
part_church_list.head(2)


part_church_list (17312, 12)


,company_name,company_location,company_address,company_website,work_phone,job_title,email,mobile_phone,first_name,last_name,source_file,denomination
0,Sherwood Baptist Church,"Huntsville, AL","6600 Old Madison Pike, Huntsville, AL 35806, U...",http://www.sherwoodbaptist.org/,256-837-0731,Pastor,robby@sherwoodbaptist.org,<NA>,Robby,Boyd,Denominations faculty C list in usa - Delive...,True
1,Valleydale Church,"Hoover, AL","2324 Valleydale Rd, Birmingham, AL 35244, Unit...",http://valleydale.org/,205-991-5282,Senior Pastor,pastormac@valleydale.org,<NA>,Mac,Brunson,Denominations faculty C list in usa - Delive...,True


In [11]:
# --- one-off: Church nondenominational workbook (hard-coded path) ---

PATH_CHURCH_NONDENOM = CHURCH_RAW_DIR / "Church nondenominational .xlsx"
if not PATH_CHURCH_NONDENOM.is_file():
    raise FileNotFoundError(PATH_CHURCH_NONDENOM)

RENAME_NONDENOM = {
    "Company Name": "company_name",
    "Church Address": "company_address",
    "Company Phone": "work_phone",
    "First Name": "first_name",
    "Last Name": "last_name",
    "Email": "email",
}

# Only one sheet in this workbook, so read it directly
part_nondenominational = pd.read_excel(PATH_CHURCH_NONDENOM, dtype="string")
part_nondenominational = rename_keep_only(part_nondenominational, RENAME_NONDENOM)
part_nondenominational = drop_fully_empty_rows(part_nondenominational)
if "first_name" in part_nondenominational.columns:
    part_nondenominational["first_name"] = part_nondenominational["first_name"].replace("", pd.NA)
else:
    part_nondenominational["first_name"] = pd.NA
if "last_name" in part_nondenominational.columns:
    part_nondenominational["last_name"] = part_nondenominational["last_name"].replace("", pd.NA)
else:
    part_nondenominational["last_name"] = pd.NA
part_nondenominational["landline_phone"] = part_nondenominational["work_phone"]
part_nondenominational["source_file"] = PATH_CHURCH_NONDENOM.name
part_nondenominational["denomination"] = False

print("part_nondenominational", part_nondenominational.shape)
part_nondenominational.head(2)


part_nondenominational (5952, 9)


,company_name,company_address,work_phone,first_name,last_name,email,landline_phone,source_file,denomination
0,103RD STREET CHURCH OF GOD,"10356 103rd St, Jacksonville, Florida, United ...",+1 904-771-3045,Roger,Bolman,roger@103churchofgod.com,+1 904-771-3045,Church nondenominational .xlsx,False
1,180 Community Church,"Commerce City, Colorado, United States",+1 720-255-6889,Jerimie,Olvera,jjerimie@180cc.church,+1 720-255-6889,Church nondenominational .xlsx,False


In [12]:
# --- one-off: USA Church workbook (hard-coded path) ---

PATH_USA_CHURCH = CHURCH_RAW_DIR / "USA Church .xlsx"
if not PATH_USA_CHURCH.is_file():
    raise FileNotFoundError(PATH_USA_CHURCH)

RENAME_USA = {
    "Church Name": "company_name",
    "Website If Avaiable": "company_website",
    "Address": "company_address",
    "Phone Number": "work_phone",
    "Pastor Email": "email",
    "Pastor Name": "__pastor_full",
    "Category": "denomination",
}

# Only one sheet in this workbook, so read it directly
part_usa_church = pd.read_excel(PATH_USA_CHURCH, dtype="string")
part_usa_church = rename_keep_only(part_usa_church, RENAME_USA)
part_usa_church = drop_fully_empty_rows(part_usa_church)
part_usa_church = add_first_last_from_full_column(part_usa_church, "__pastor_full")
part_usa_church["landline_phone"] = part_usa_church["work_phone"]
part_usa_church["source_file"] = PATH_USA_CHURCH.name
part_usa_church["denomination"] = part_usa_church["denomination"].map({"Non-denominational church": False, "Church": True})

print("part_usa_church", part_usa_church.shape)
part_usa_church.head(2)


part_usa_church (3685, 10)


,company_name,company_website,company_address,work_phone,email,denomination,first_name,last_name,landline_phone,source_file
0,1 Life Church,https://1lifechurchcleburne.com/,"710 W Kilpatrick St, Cleburne, TX 76033",(817) 526-1826,info@1lifechurchcleburne.com,False,Efrain,Villarreal,(817) 526-1826,USA Church .xlsx
1,1200 Church,http://www.1200.church/,"1200 W Tharpe St, Tallahassee, FL 32303",(850) 224-0914,1200church@gmail.com,True,Anthony,Battle,(850) 224-0914,USA Church .xlsx


In [13]:
# --- one-off: stack parts + fill TARGET_COLS ---

final_df = pd.concat(
    [part_church_list, part_nondenominational, part_usa_church],
    ignore_index=True,
)

# Normalize control/newline noise before parsing and exporting.
final_df = normalize_string_columns(final_df)

final_df = add_parsed_address_columns(final_df)
final_df = clean_company_address_components(final_df)

for c in TARGET_COLS:
    if c not in final_df.columns:
        final_df[c] = pd.NA

final_df = final_df[TARGET_COLS]
final_df["primary_naics"] = "813110"

# Re-normalize after all shaping, then standardize phones/emails for Excel/CSV consumers.
final_df = normalize_string_columns(final_df)
final_df = clean_contact_columns(final_df)
final_df = final_df.drop_duplicates().reset_index(drop=True)

# Set IDs after all other shaping so IDs match final exported row order.
final_df["record_id"] = "c-" + (final_df.index + 1).astype(str)

print("final_df shape:", final_df.shape)
print("columns match target:", list(final_df.columns) == TARGET_COLS)
final_df


final_df shape: (26706, 75)
columns match target: True


,source_file,denomination,record_id,first_name,last_name,job_title,email,mobile_phone,company_linkedin,facebook,...,voting_performance_even_year_primary,voting_performance_even_year_general_and_primary,voting_performance_minor_election,primary_n_of_4,general_2024,primary_2024,general_2022,primary_2022,general_2020,primary_2020
0,Denominations faculty C list in usa - Delivery...,True,c-1,Robby,Boyd,Pastor,robby@sherwoodbaptist.org,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,Denominations faculty C list in usa - Delivery...,True,c-2,Mac,Brunson,Senior Pastor,pastormac@valleydale.org,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,Denominations faculty C list in usa - Delivery...,True,c-3,Trey,Waldrop,Lead Pastor,churchoffice@fbctallassee.com,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,Denominations faculty C list in usa - Delivery...,True,c-4,Rose,Leblanc,Pastor,maosophearath5@gmail.com,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,Denominations faculty C list in usa - Delivery...,True,c-5,Tommy,Poole,Pastor,tommy.poole@wallawalla.edu,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26701,USA Church .xlsx,True,c-26702,Jonathan,Meyer,<NA>,zlmv@ccwip.net,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
26702,USA Church .xlsx,True,c-26703,Karin,Albaugh,<NA>,info@zionlutheranfl.com,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
26703,USA Church .xlsx,False,c-26704,Jesse,Terasaki,<NA>,info@zoedallas.org,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
26704,USA Church .xlsx,True,c-26705,Aleksey,Zhuravlev,<NA>,info@izavet.org,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [14]:
final_df.to_csv("csv_outputs/church_data_20260404.csv", index=False, encoding="utf-8-sig")

abstracted_columns = ["record_id", "first_name", "last_name", "mobile_phone", "company_city", "company_state", "company_zipcode"]
abstracted_df = final_df[abstracted_columns]
abstracted_df.to_csv("csv_outputs/church_data_abstracted_20260404.csv", index=False, encoding="utf-8-sig")